# 0. Packages

In [1]:
import pandas as pd
import os

# 1. Relevant scraping information and data

**NOTE**: *the scraping has been undertaken considering that we will do a difference-in-differences analysis of the price impact of the Mobile World Congress in Barcelona (which lasts from the 3rd of March to the 6th of March in 2025). This is the reason why only the first weeks of each month have been extracted, both for Barcelona and Madrid.*

About the scraping:
1. The scraping of all of the information has been done on the 25th January 2025.
2. The dates comprised where accommodation information has been extracted are:
    - Start date: February 2025.
    - End date: April 2026.
3. For each month, information has been scraped for the 1st to the 8th.
4. If you want to try out that the `booking_scraping.py` works, the range of months between the start and the end must include, at least, a month in the future. E.g., if we are in February 2025, you must include at least March 2026 as the end month.
5. The duration of the scraping depends (mainly) on your connectivity. If your connection is not very stable, increase the parameter `time_sleep` in the script `booking_scraping.py`.
6. With a `time_sleep` of 2 seconds, the setting described above, 6037 distinct accommodations overall and a relatively stable connection the scraping has lasted for, approximately, 5 hours (1 hours 30 minutes the extraction of the general accommodation information and 3 hours 30 minutes the extraction of the descriptions of 6037 accommodations through requests).

About where to locate the data and how to extract useful information from it:
1. All the data that has been extracted is located in the `files` folder within the repo.
2. Inside the `files` folder, the `general_results` folder contains .csv where each file is general accommodation data for one place and one month (the range extracted is indicated in the file itself).
    - Features included: name of the accommodation, price, rating and neighborhood of the accommodation. The neighborhood was included considering the possibility that accommodation names were not unique, but it seems that they have to be unique according to Booking policies.
    - Data is not in the same file in order to avoid data loss while doing the scraping (if we saved all of the data only in the end, the progress would have been lost).
    - How to analyze it: I would recommend joining the whole .csv files into one or two separate data frames (depending on whether you want to include Barcelona and Madrid in the same data frame, which should not be a problem given that the neighborhood column includes the city name), where in both cases you should tag (i.e., add an additional column) the week for which the information has been extracted (i.e., 2025-05-01_2025-05-08, in yyyy-mm-dd format). But whatever you do, don't remove the original data.
    - This data extraction has been done with Selenium.
3. On the other hand, also inside the `files` folder, the `accommodation_descriptions` folder contains the file the file `descriptions_6037.csv` with the descriptions of all of the unique accommodations for the whole period that has been scraped.
    - Features included: name of the accommodation, neighborhood, URL (extracted when scraping the general results, but it is not useful anymore if we have the description - you can drop it in the analysis) and the description (in English! Can also be done in any other language, but this was the preferred choice for obvious reasons). 
    - When doing the scraping, it progressively saved the data as it increased the number of descriptions scraped. But in the end I have only kept the file with all the descriptions, `descriptions_6037.csv`.
    - For matching the descriptions with the general information of the accommodations, you can do it with the name of the accommodation. If the name of the accommodation was not enough to uniquely identify an accommodation, you can also include the neighborhood. 

# 2. Checking the `general_results` files

In [2]:
df_descriptions = pd.read_csv('files_eng/accommodation_descriptions/descriptions_6037.csv')
df_descriptions

,hotel_name,neighborhood,url,description
0,Relais & Châteaux Hotel Orfila,"Chamberi, Madrid",https://www.booking.com/hotel/es/relais-chatea...,Hotel Orfila is situated in Madrid’s Chamberí ...
1,B&B HOTEL Madrid Carabanchel,"Carabanchel, Madrid",https://www.booking.com/hotel/es/b-amp-b-madri...,Conveniently set in the Carabanchel district o...
2,Apartamentos Adelfas,"Retiro, Madrid",https://www.booking.com/hotel/es/apartamentos-...,These modern and stylish apartments offer free...
3,Elba Madrid Alcalá,"San Blas, Madrid",https://www.booking.com/hotel/es/velada-madrid...,Elba Madrid Alcalá is ideally located between ...
4,Apartamentos Juan Bravo,"Salamanca, Madrid",https://www.booking.com/hotel/es/apartamentos-...,The Juan Bravo Apartments are situated in the ...
...,...,...,...,...
6032,For You Rentals Cómodo apartamento de un dormi...,"Ciudad Lineal, Madrid",https://www.booking.com/hotel/es/for-you-renta...,"Located in Madrid, 5.9 km from El Retiro Park ..."
6033,For Your Rentals Amplio y luminoso apartamento...,"Usera, Madrid",https://www.booking.com/hotel/es/amplio-y-lumi...,"Located in Madrid, the recently renovated For ..."
6034,For You Rentals Acogedor Apartamento TEMPORAL ...,"Madrid City Center, Madrid",https://www.booking.com/hotel/es/for-you-renta...,For You Rentals Acogedor Apartamento TEMPORAL ...
6035,Apartamento Bernabeu Loft en Madrid,"Tetuan, Madrid",https://www.booking.com/hotel/es/mit-house-ber...,Apartamento Bernabeu Loft en Madrid is located...


In [3]:
df_descriptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6037 entries, 0 to 6036
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   hotel_name    6037 non-null   object
 1   neighborhood  6037 non-null   object
 2   url           6037 non-null   object
 3   description   6037 non-null   object
dtypes: object(4)
memory usage: 188.8+ KB


There are no missing values!! Also, from an exploratory analysis of the data frame it seems that all of the descriptions are correctly matched to the hotel.

In [6]:
df_descriptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8237 entries, 0 to 8236
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   hotel_name    5979 non-null   object
 1   neighborhood  5979 non-null   object
 2   url           5979 non-null   object
 3   description   5900 non-null   object
dtypes: object(4)
memory usage: 257.5+ KB


In [8]:
df = pd.DataFrame([[0, 2, 3], [0, 4, 1], [10, 20, 30]], columns=['A', 'B', 'C'])
df

,A,B,C
0,0,2,3
1,0,4,1
2,10,20,30


In [12]:
print(df.at[2, 'B'])

20


In [18]:
url_data_path = "files/accommodation_urls"

# Step 1: import all URL .csv into dataframes in a loop
dataframes = []
# os.listdir(folder_path) lists all files in the specified folder
for file in os.listdir(url_data_path):
    if file.endswith(".csv"):
        file_path = os.path.join(url_data_path, file)
        df_url = pd.read_csv(file_path)
        dataframes.append(df_url)

# Step 2: concatenate all dataframes along the rows (indexes, axis = 0)
df_url_combined = pd.concat(dataframes, axis=0, ignore_index=True)

# Step 3: keep only unique values among a subset of columns (so that the extraction
# of descriptions is more efficient)
df_url_unique = df_url_combined.drop_duplicates(subset=['hotel_name']).reset_index(drop = True)

In [19]:
df_url_unique

,hotel_name,neighborhood,url
0,Relais & Châteaux Hotel Orfila,"Chamberí, Madrid",https://www.booking.com/hotel/es/relais-chatea...
1,B&B HOTEL Madrid Carabanchel,"Carabanchel, Madrid",https://www.booking.com/hotel/es/b-amp-b-madri...
2,Market Martinez de la Riva I,"Puente de Vallecas, Madrid",https://www.booking.com/hotel/es/apartamentos-...
3,Apartamentos Adelfas,"Retiro, Madrid",https://www.booking.com/hotel/es/apartamentos-...
4,Apartamentos Villablino Arturo Soria,"Ciudad Lineal, Madrid",https://www.booking.com/hotel/es/apartamentos-...
...,...,...,...
5974,For You Rentals Acogedor Apartamento TEMPORAL ...,"Centro de Madrid, Madrid",https://www.booking.com/hotel/es/for-you-renta...
5975,For You Rentals Encantador apartamento TEMPORA...,"Centro de Madrid, Madrid",https://www.booking.com/hotel/es/for-you-renta...
5976,Elegant Apartment in the Heart of Madrid 100m²...,"Centro de Madrid, Madrid",https://www.booking.com/hotel/es/apartment-cal...
5977,For You Rentals Hermoso apartamento TEMPORAL P...,"Centro de Madrid, Madrid",https://www.booking.com/hotel/es/habitaciones-...
